<a href="https://colab.research.google.com/github/ElenaLuchnikova/Analyzes/blob/main/%D0%9A%D0%BE%D0%B3%D0%BE%D1%80%D1%82%D0%BD%D1%8B%D0%B9_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_Retention_SQLight3_%D0%B8%D0%BD%D1%82%D0%B5%D1%80%D0%BD%D0%B5%D1%82_%D0%BC%D0%B0%D0%B3%D0%B0%D0%B7%D0%B8%D0%BD_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Когортный анализ** — это метод анализа данных, который позволяет исследовать поведение и характеристики групп людей (когорт), объединённых по определённому признаку, например, по времени начала использования продукта, даты регистрации, даты первой покупки и т.д.

Основная идея — разделить пользователей или объекты на когорты и отслеживать их поведение со временем. Это помогает понять, как изменяются показатели (например, удержание, активность, доходы) у разных групп и выявить тренды или закономерности.

У нас есть данные о продажах интернет-магазина косметики и бытовой химии за 2023-2025 годы. Известно, что с января по июнь 2024 проводились кампании по привлечению клиентов, в том числе скидки на повторную покупку с ограничением по времени - первая неделя. Нужно проанализировать эффективность этих кампаний в плане удержания клиентов, выявить лучшие когорты и дать рекомендации по дальнейшей маркетинговой политике.

Выполним с помощью SQL когортный анализ Retention (удержание) и проанализируем поведение когорт.

In [3]:
import pandas as pd
import numpy as np
import sqlite3
from google.colab import drive
import seaborn as sns
import matplotlib.pyplot as plt



In [4]:
#df = pd.read_csv('/content/drive/MyDrive/data_store.csv',encoding='1251',sep=';')

In [5]:
df=pd.read_csv('https://raw.githubusercontent.com/ElenaLuchnikova/Datasets/refs/heads/main/data_store.csv', encoding='1251', sep=';')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52286 entries, 0 to 52285
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   user    52286 non-null  int64 
 1   dt      52286 non-null  object
 2   sale    52286 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.2+ MB


In [7]:
df.head()

,user,dt,sale
0,1,27.02.2024,2763
1,1,28.02.2024,5872
2,1,01.03.2024,540
3,1,02.03.2024,3535
4,1,04.03.2024,1256


In [8]:

df['dt']=pd.to_datetime(df['dt'], format="%d.%m.%Y")

In [9]:
# Создаём подключение к временной базе данных в памяти
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

In [10]:
# Создаем таблицу с учетом типов данных
cursor.execute('''
CREATE TABLE data_store  (

    user INT,

    dt DATE,

    sale REAL

)
''')

In [11]:
# Вставляем данные из DataFrame в таблицу
df.to_sql('data_store', conn, if_exists='append', index=False)

52286

In [12]:
query = "select *  from data_store limit 5"
result = pd.read_sql_query(query, conn)
print(result)

   user                   dt    sale
0     1  2024-02-27 00:00:00  2763.0
1     1  2024-02-28 00:00:00  5872.0
2     1  2024-03-01 00:00:00   540.0
3     1  2024-03-02 00:00:00  3535.0
4     1  2024-03-04 00:00:00  1256.0


# Когортный анализ Retention

**Выполним когортный анализ Retention:**


1)для каждого клиента определим его когорту (в данном случае - месяц и год самой первой покупки)

2)сформируем таблицу, в которой выделим периоды: 1, 2, 3, 4, 5, 6, 7, 14, 30, 60, 90, 180, 270 и 360 дней с момента первой покупки и посчитаем, сколько клиентов вернулось в каждый из дней


In [13]:
query='''
with t1 as (
	select
		dt,
		user,
		sale,
		-- Формируем когорту, находя самую первую покупку для каждого клиента
		-- и извлекая из нее месяц и год
		strftime('%Y-%m',
            first_value(dt) over(partition by user order by dt)
        ) as cohort,
        first_value(dt) over(partition by user order by dt) as first_day,
        julianday(dt) - julianday(first_value(dt) over(partition by user order by dt)) as diff

	from data_store
	where dt between '2024-01-01' and '2025-01-01'),

  t2 as (select cohort,
  diff,
  count(distinct user) as cnt
  from t1
  where diff in (0,1,2,3,4,5,6,7, 14,30,60,90,180,270,360)
  group by cohort, diff
  order by cohort, diff)



  select cohort,
  max(case when diff=0 then cnt end) as "0",
  max(case when diff=1 then cnt end) as "1",
  max(case when diff=2 then cnt end) as "2",
  max(case when diff=3 then cnt end) as "3",
  max(case when diff=4 then cnt end) as "4",
  max(case when diff=5 then cnt end) as "5",
  max(case when diff=6 then cnt end) as "6",
  max(case when diff=7 then cnt end) as "7",
  max(case when diff=14 then cnt end) as "14",
  max(case when diff=30 then cnt end) as "30",
  max(case when diff=60 then cnt end) as "60",
  max(case when diff=90 then cnt end) as "90",
  max(case when diff=180 then cnt end) as "180",
  max(case when diff=270 then cnt end) as "270",
  max(case when diff=360 then cnt end) as "360"
  from t2
  group by cohort
  '''

# Получаем результат
result = pd.read_sql_query(query, conn)
# Выводим результат
print(result)

    cohort    0      1      2      3      4      5      6      7    14    30  \
0  2024-01  811  149.0  150.0  130.0  137.0  131.0  117.0  130.0  81.0  85.0   
1  2024-02  208   66.0   73.0   73.0   68.0   63.0   63.0   56.0  37.0  43.0   
2  2024-03  170   67.0   68.0   74.0   82.0   74.0   72.0   65.0  27.0  20.0   
3  2024-04  124   51.0   57.0   55.0   56.0   48.0   49.0   48.0  24.0  27.0   
4  2024-05  152   78.0   75.0   73.0   83.0   74.0   71.0   64.0  40.0  30.0   
5  2024-06  108   59.0   59.0   58.0   53.0   51.0   51.0   60.0  28.0  18.0   
6  2024-07    3    NaN    NaN    NaN    NaN    NaN    NaN    NaN   NaN   NaN   
7  2024-09    1    NaN    NaN    NaN    NaN    NaN    NaN    NaN   NaN   NaN   

     60    90   180  270   360  
0  40.0  35.0  12.0  NaN  None  
1  15.0  19.0   3.0  1.0  None  
2  24.0  14.0   1.0  NaN  None  
3  10.0  11.0   2.0  NaN  None  
4  17.0  14.0   4.0  NaN  None  
5  22.0  10.0   5.0  NaN  None  
6   NaN   NaN   NaN  NaN  None  
7   NaN   NaN  

In [14]:
res=pd.DataFrame(result)
res.set_index(res.columns[0], inplace=True)

In [15]:
res

,0,1,2,3,4,5,6,7,14,30,60,90,180,270,360
cohort,,,,,,,,,,,,,,,
2024-01,811,149.0,150.0,130.0,137.0,131.0,117.0,130.0,81.0,85.0,40.0,35.0,12.0,NaN,None
2024-02,208,66.0,73.0,73.0,68.0,63.0,63.0,56.0,37.0,43.0,15.0,19.0,3.0,1.0,None
2024-03,170,67.0,68.0,74.0,82.0,74.0,72.0,65.0,27.0,20.0,24.0,14.0,1.0,NaN,None
2024-04,124,51.0,57.0,55.0,56.0,48.0,49.0,48.0,24.0,27.0,10.0,11.0,2.0,NaN,None
2024-05,152,78.0,75.0,73.0,83.0,74.0,71.0,64.0,40.0,30.0,17.0,14.0,4.0,NaN,None
2024-06,108,59.0,59.0,58.0,53.0,51.0,51.0,60.0,28.0,18.0,22.0,10.0,5.0,NaN,None
2024-07,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2024-09,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [19]:
styled_res = res.style.background_gradient(cmap='YlGnBu',vmin=0, vmax=812).format( precision=0)
styled_res

,0,1,2,3,4,5,6,7,14,30,60,90,180,270,360
cohort,,,,,,,,,,,,,,,
2024-01,811,149,150,130,137,131,117,130,81,85,40,35,12,nan,None
2024-02,208,66,73,73,68,63,63,56,37,43,15,19,3,1,None
2024-03,170,67,68,74,82,74,72,65,27,20,24,14,1,nan,None
2024-04,124,51,57,55,56,48,49,48,24,27,10,11,2,nan,None
2024-05,152,78,75,73,83,74,71,64,40,30,17,14,4,nan,None
2024-06,108,59,59,58,53,51,51,60,28,18,22,10,5,nan,None
2024-07,3,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,None
2024-09,1,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,None


* Когорты Июля и Сентября включают в себя 1-3 покупателя (разовые клиенты, мало данных), поэтому мы их анализировать не будем. За август нет данных.
*  Учитывая специфику магазина (косметика и БХ - товары не каждодневного спроса), можем отметить, что в первую неделю возвращается довольно большое количество клиентов по всем когортам. Можно сделать вывод, что акция на повторные покупки в течение первой недели довольно эффективна.

* Но после первой недели видим постепенное снижение возврата покупателей в долгосрочной перспективе по всем когортам - на 180й день возвращается только 1-12 покупателей. К 270 дню почти все покупатели отваливаются.  

*  Самая большая когорта - Январская когорта (811 клиентов). Скорее всего в этом месяце проходили новогодние распродажи, акции и проводилось много рекламных кампаний по привлечению новых покупателей. В первую неделю вернулось довольно много клиентов для данной ниши. Но затем покупатели постепенно начали отваливаться, мало повторных покупок. Большинству клиентов этой когорты не интересен магазин и продукция, много разовых покупателей, привлеченных большой скидкой (возможно, привлекается дешевый и некачественный трафик аудитории).


*   В остальных когортах количество привлеченных покупателей заметно меньше.
 Поведение клиентов аналогично Январской когорте -  также после первой недели начинается спад покупок и в итоге клиенты уходят. Хотя в процентном соотношении отток покупателей не такой большой, как в когорте Января, но к 270 дню клиенты отваливаются.

*  Наиболее стабильными когортами оказались Апрельская, Майская и  Июньская. В самом начале привлечено было не так много клиентов (124, 152 и 108). Но соотношение сохранившихся покупателей к изначальному числу (по месяцам) лучше, чем в других когортах. Возможно, лучше была настроена рекламная компания, которая привлекла больше именно постоянных клиентов, которым понравился сам магазин и продукция. Следует проанализировать настройки майских и июньских рекламных компаний, понять, почему они были успешны в плане формирования ядра постоянных покупателей. Посмотреть, какие рекламные каналы использовались.

#Общие выводы

**У магазина большие проблемы с удержанием покупателей. Повторных покупок к 270 дню почти не остается. Не формируется ядро постоянных клиентов.**

 **Но есть есть успехи в привлечении и краткосрочном удержании: магазину удается успешно привлекать новых клиентов и стимулировать их на повторные покупки в первую неделю после первого касания. Эту тактику следует периодически повторять в течение всего lifetime клиента.**

#Выводы по когортам
**В плане привлечения клиентов лучше себя показала когорта Января. А если говорить об удержании - то когорты Апреля, Мая и Июня.**

#Рекомендации

**Следует поработать с качеством привлекаемых клиентов, проанализировать рекламные каналы и настройки рекламы (за пример взять более успешные в плане удержания когорты Апреля-Июня).**

 **Провести работу с лояльностью покупателей и стимулированием повторных покупок. Я бы порекомендовала расширить программы лояльности: бонусную программу, кэшбэк за покупки, рассылки, акции и скидки к праздникам, персональные подборки товаров. Желательно периодически повторять акции на повторные покупки (с ограничением по времени) или  на бесплатную доставку для повторной покупки.**

**Также важно проверить технические моменты: может быть у магазина неудобный сайт, сложности с оплатой, неудобный поиск товаров.**

**Нелишним будет проанализировать цены у магазинов-конкурентов (возможно новые клиенты покупают первый раз только благодаря большой скидке за первый заказ, а совершать повторные покупки для них дорого и они уходят к конкурентам).**

**Конечные выводы об эффективности маркетинговых кампаний можно сделать только после проведения [Когортного анализа выручки или прибыли (LTV)](https://colab.research.google.com/drive/1EnRt64Z4dmYByWZ1rz3y_9kUryUh5k6x?usp=sharing), чтобы понять насколько они принесли пользу в финансовом плане. Человек может покупать часто, но на маленькие суммы, а для магазина важен именно доход от клиента.**







